# Dịch `merged_products.csv` trên Google Colab

Dùng NVIDIA API dịch **`title`**, **`description`**, **`category`**, **`tags`** sang tiếng Việt — **tên cột giữ nguyên**. Giữ nguyên **`price`**, `brand`, URL, rating…

**Cần:** `NVIDIA_API_KEY` · internet · **không cần GPU**

### Quy trình nhanh
1. Mở notebook trên Colab → **Runtime → Run all** (hoặc chạy từng cell)
2. Có **code project** (clone GitHub hoặc upload zip)
3. Thêm Secret **`NVIDIA_API_KEY`** (🔑)
4. Upload **`merged_products.csv`**
5. Chạy dịch thử `--limit 5` → OK thì chạy full → tải `merged_products_vi.csv`

## 1. Lấy code project lên Colab

Chọn **một** cách:

| Cách | Khi nào dùng |
|------|----------------|
| **A. GitHub** | Repo đã push — sửa `REPO_URL` rồi chạy cell clone |
| **B. Upload zip** | Chưa có GitHub — nén cả folder project thành `.zip`, chạy cell upload zip |
| **C. Mở từ GitHub** | File → Open notebook → GitHub → chọn `colab.ipynb` (code đã có sẵn) |

In [1]:
PROJECT_DIR = "/content/llm_provider_benchmarking"

# --- Cách A: clone (sửa URL repo của bạn) ---
USE_GIT_CLONE = True
REPO_URL = "https://github.com/PhamMinhDan/llm_provider_benchmarking.git"  # ← sửa

import os
import subprocess

if USE_GIT_CLONE and not os.path.isdir(PROJECT_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR], check=True)
    print("Đã clone:", PROJECT_DIR)
elif os.path.isdir(PROJECT_DIR):
    print("Đã có thư mục:", PROJECT_DIR)
else:
    print("Chưa có project — dùng cell upload zip bên dưới hoặc bật USE_GIT_CLONE.")

Đã clone: /content/llm_provider_benchmarking


In [ ]:
# --- Cách B: upload zip toàn bộ project (bỏ qua nếu đã clone) ---
import os
import zipfile
from google.colab import files

PROJECT_DIR = "/content/llm_provider_benchmarking"

if not os.path.isfile(os.path.join(PROJECT_DIR, "nvidia_provider", "translate_products.py")):
    print("Chọn file .zip chứa folder llm_provider_benchmarking (hoặc nội dung project)...")
    uploaded = files.upload()
    for name, data in uploaded.items():
        if not name.lower().endswith(".zip"):
            continue
        with open(name, "wb") as f:
            f.write(data)
        with zipfile.ZipFile(name) as z:
            z.extractall("/content")
        print("Đã giải nén:", name)
        break
else:
    print("Code project đã sẵn sàng, bỏ qua upload zip.")

In [2]:
import os
import sys

PROJECT_DIR = "/content/llm_provider_benchmarking"

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Không thấy {PROJECT_DIR}. Clone GitHub hoặc upload zip project."
    )

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("cwd:", os.getcwd())
print("translate_products:", os.path.isfile("nvidia_provider/translate_products.py"))

cwd: /content/llm_provider_benchmarking
translate_products: True


## 2. Cài thư viện

In [3]:
%pip install -q openai python-dotenv

## 3. API key NVIDIA

1. Thanh trái → **🔑 Secrets** → **Add new secret**
2. Name: `NVIDIA_API_KEY` (đúng chữ, phân biệt hoa thường)
3. Dán API key → **bật** quyền notebook cho secret
4. Chạy cell dưới

In [4]:
import os
from google.colab import userdata

os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
key = os.environ["NVIDIA_API_KEY"]
print(f"API key OK — {len(key)} ký tự, bắt đầu: {key[:12]}...")

API key OK — 70 ký tự, bắt đầu: nvapi-mbUjgK...


## 4. Upload `merged_products.csv`

Chạy cell dưới và chọn file CSV (~2000 dòng).  
Hoặc từ Drive: mount Drive rồi `!cp ...` (xem comment trong cell).

In [5]:
import os
import shutil
from google.colab import files

CSV_PATH = os.path.join(PROJECT_DIR, "merged_products.csv")

if not os.path.isfile(CSV_PATH):
    print("Chọn file merged_products.csv trên máy...")
    uploaded = files.upload()
    for name in uploaded:
        dest = CSV_PATH if name.endswith(".csv") else os.path.join(PROJECT_DIR, name)
        shutil.move(name, dest)
        print("Đã lưu:", dest)
else:
    print("CSV đã có:", CSV_PATH)

# Tùy chọn — Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/merged_products.csv" "{CSV_PATH}"

CSV đã có: /content/llm_provider_benchmarking/merged_products.csv


## 5a. Dịch thử (5 sản phẩm)

Kiểm tra API key + chất lượng dịch trước khi chạy full.

In [6]:
!python nvidia_provider/translate_products.py \
  --input merged_products.csv \
  --output merged_products_vi.csv \
  --limit 5 \
  --delay 0.5

Probe 19 model trong 1.5s...

  ✗ qwen/qwen3-next-80b-a3b-thinking                     [HTTP 410]
  ✓ mistralai/ministral-14b-instruct-2512                1.293s
  ✗ qwen/qwq-32b                                         [HTTP 404]
  ✓ openai/gpt-oss-120b                                  1.329s
  ✗ z-ai/glm5                                            [HTTP 410]
  ✗ meta/llama-3.1-405b-instruct                         [HTTP 404]
  ✓ nvidia/nemotron-3-nano-30b-a3b                       1.279s
  ✗ nvidia/nvidia-nemotron-nano-9b-v2                    [HTTP 400]
  ✗ z-ai/glm4.7                                          [HTTP 410]
  ✓ meta/llama-3.3-70b-instruct                          1.325s
  ✓ openai/gpt-oss-20b                                   1.273s
  ✗ bytedance/seed-oss-36b-instruct                      [HTTP 400]
  ✓ qwen/qwen3.5-122b-a10b                               1.411s
  ✗ mistralai/mistral-large-3-675b-instruct-2512         [không phản hồi trong 1.5s]
  ✗ nvidia/nemotron-3-nan

In [7]:
# Xem nhanh 5 dòng đã dịch
import pandas as pd
df = pd.read_csv("merged_products_vi.csv")
cols = ["product_id", "brand", "title", "description", "category", "tags"]
display(df[cols].head())

,product_id,brand,title,description,category,tags
0,B09NQJFRW6,Saucony,Giày chạy bộ Saucony Kinvara 13 cho nam,"Kinvara 13 là mẫu giày nhẹ nhất của Saucony, m...","Quần áo, Giày dép & Trang sức > Nam > Giày dép...","Quần áo, Giày dép & Trang sức | Nam | Giày dép..."
1,B0074TRKFI,Kishigo,Áo phản quang an toàn Kishigo Premium Black Se...,Áo Kishigo Premium Black Series Heavy Duty lý ...,Dụng cụ & Cải tạo nhà cửa > An toàn & Bảo mật ...,Dụng cụ & Cải tạo nhà cửa | An toàn & Bảo mật ...
2,B07V5LK5J3,TWINSLUXES,Đèn nắp cột năng lượng mặt trời TWINSLUXES ngo...,Đèn nắp cột năng lượng mặt trời TWINSLUXES chố...,Công cụ & Cải tạo nhà cửa > Đèn & Quạt trần > ...,Lắp đặt nhanh chóng và dễ dàng | Khả năng chốn...
3,B00080QHMM,Accutire,Bộ đo áp suất lốp xe Accutire MS-4021B số kỹ t...,Thiết bị đo áp suất lốp bền bỉ với thiết kế ch...,Ô tô > Dụng cụ & Thiết bị > Dụng cụ lốp & bánh...,"Thiết kế chắc chắn, bền bỉ|Đầu đo góc nghiêng,..."
4,B0CNZ6ZV14,SAURA LIFE SCIENCE,SAURA LIFE SCIENCE Dầu gội thảo dược Adivasi A...,"Sản phẩm kết hợp độc đáo này giúp nuôi dưỡng, ...",Làm đẹp > Chăm sóc tóc > Dầu gội,Giảm rụng tóc | Kích thích mọc tóc | Dưỡng ẩm ...


## 5b. Dịch toàn bộ (~2000 dòng)

- Ước tính **2–6 giờ** (tùy API).
- **Resume:** chạy lại cell này nếu bị ngắt — dùng file checkpoint (tự tạo cạnh file output).
- Để chạy lại từ đầu: thêm `--no-resume` trong lệnh.

⚠️ Giữ tab Colab mở; Colab có thể ngắt session sau ~90 phút không tương tác.

In [ ]:
!python nvidia_provider/translate_products.py \
  --input merged_products.csv \
  --output merged_products_vi.csv \
  --delay 0.5

Probe 19 model trong 1.5s...

  ✓ openai/gpt-oss-20b                                   0.853s
  ✓ nvidia/nemotron-3-nano-30b-a3b                       0.877s
  ✗ qwen/qwq-32b                                         [HTTP 404]
  ✗ z-ai/glm5                                            [HTTP 410]
  ✗ z-ai/glm4.7                                          [HTTP 410]
  ✓ qwen/qwen3.5-122b-a10b                               1.026s
  ✗ qwen/qwen3-next-80b-a3b-thinking                     [HTTP 410]
  ✗ nvidia/nvidia-nemotron-nano-9b-v2                    [HTTP 400]
  ✗ bytedance/seed-oss-36b-instruct                      [HTTP 400]
  ✗ meta/llama-3.1-405b-instruct                         [HTTP 404]
  ✓ meta/llama-3.3-70b-instruct                          1.209s
  ✓ openai/gpt-oss-120b                                  1.020s
  ✓ nvidia/nemotron-3-nano-omni-30b-a3b-reasoning        0.905s
  ✓ mistralai/ministral-14b-instruct-2512                0.883s
  ✗ mistralai/mistral-large-3-675b-instruct-25

## 6. Tải file kết quả về máy

In [ ]:
from google.colab import files
import os

out = "merged_products_vi.csv"
if os.path.isfile(out):
    files.download(out)
else:
    print("Chưa có file output — chạy bước dịch trước.")

---
## (Tùy chọn) Chạy benchmark voicebot

Probe model + stream context mẫu — **không** liên quan bước dịch CSV.

In [ ]:
# from nvidia_provider.benchmark import main
# main()

## Xử lý lỗi

| Lỗi | Cách xử lý |
|-----|------------|
| Secret / `NVIDIA_API_KEY` | Thêm secret 🔑, bật quyền notebook |
| `HTTP 401` | Key sai hoặc hết hạn |
| `Không có model phản hồi probe` | Thử `--skip-probe` hoặc `--model qwen/qwen3-next-80b-a3b-instruct` |
| Session disconnect | Chạy lại cell **5b** (resume tự động) |
| `CSV tồn tại: False` | Chạy lại cell upload CSV |

**Bảo mật:** Không share notebook công khai nếu dán key trực tiếp vào code.